In [0]:
import yaml
import os
from pyspark.sql import functions as F

repo_path = os.getcwd().split("/notebooks/")[0]

config_path = f"{repo_path}/configs/env/dev.yml"

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

CATALOG       = cfg["catalog"]
SILVER_SCHEMA = cfg["silver_schema"]
GOLD_SCHEMA   = cfg["gold_schema"]

SILVER = f"{CATALOG}.{SILVER_SCHEMA}.spotify_tracks"
G      = f"{CATALOG}.{GOLD_SCHEMA}"

print(f"Reading from : {SILVER}")
print(f"Writing to   : {G}.*")

In [0]:
dims = ["dim_genre", "dim_artist", "dim_album", "dim_audio_features"]

print(f"{'Dimension':<30} {'Rows':>10}")
print("-" * 42)
all_ok = True
for dim in dims:
    try:
        count = spark.table(f"{G}.{dim}").count()
        status = "✓" if count > 0 else "✗ EMPTY"
        if count == 0:
            all_ok = False
        print(f"{dim:<30} {count:>10,}  {status}")
    except Exception as e:
        print(f"{dim:<30}  ERROR: {e}")
        all_ok = False

if not all_ok:
    raise Exception("One or more dimension tables are missing or empty. Run 01_build_dimensions first.")

In [0]:
df_silver = spark.table(SILVER)

print(f"Silver rows         : {df_silver.count():,}")
print(f"Distinct track_ids  : {df_silver.select('track_id').distinct().count():,}")
print(f"Multi-artist tracks : {df_silver.filter(F.size('artists') > 1).count():,}")

display(
    df_silver
    .select("track_id", "track_name", "artists", "album_name",
            "track_genre", "popularity", "duration_ms", "explicit")
    .limit(5)
)

In [0]:
spark.sql(f"""
    WITH exploded AS (
        SELECT
            s.track_id,
            TRIM(artist_col.col) AS artist_name,
            s.album_name,
            s.track_genre
        FROM {SILVER} s
        LATERAL VIEW EXPLODE(s.artists) artist_col
    )
    SELECT
        COUNT(*)                                                AS total_exploded_rows,
        SUM(CASE WHEN da.artist_id  IS NULL THEN 1 ELSE 0 END) AS unmatched_artists,
        SUM(CASE WHEN dal.album_id  IS NULL THEN 1 ELSE 0 END) AS unmatched_albums,
        SUM(CASE WHEN dg.genre_id   IS NULL THEN 1 ELSE 0 END) AS unmatched_genres,
        SUM(CASE WHEN daf.audio_id  IS NULL THEN 1 ELSE 0 END) AS unmatched_audio
    FROM exploded e
    LEFT JOIN {G}.dim_artist         da  ON e.artist_name = da.artist_name
    LEFT JOIN {G}.dim_album          dal ON TRIM(e.album_name) = dal.album_name
    LEFT JOIN {G}.dim_genre          dg  ON LOWER(TRIM(e.track_genre)) = dg.genre_name
    LEFT JOIN {G}.dim_audio_features daf ON e.track_id = daf.track_id
""").show()

In [0]:
spark.sql(f"""
    MERGE INTO {G}.fact_tracks AS target
    USING (
        -- CTE 1: explode artists array — one row per (track, artist)
        WITH exploded AS (
            SELECT
                s.track_id,
                s.track_name,
                TRIM(artist_col.col) AS artist_name,
                s.album_name,
                s.track_genre,
                s.popularity,
                s.duration_ms,
                s.explicit
            FROM {SILVER} s
            LATERAL VIEW EXPLODE(s.artists) artist_col
            WHERE s.track_id IS NOT NULL
        ),

        -- CTE 2: resolve surrogate FKs and compute derived measures
        resolved AS (
            SELECT
                e.track_id,
                e.track_name,
                e.popularity,
                e.duration_ms,
                -- Convert milliseconds to MM:SS display format
                CONCAT(
                    CAST(FLOOR(e.duration_ms / 60000) AS STRING),
                    ':',
                    LPAD(CAST(FLOOR((e.duration_ms % 60000) / 1000) AS STRING), 2, '0')
                )                       AS duration_formatted,
                e.explicit,
                da.artist_id,
                dal.album_id,
                dg.genre_id,
                daf.audio_id
            FROM exploded e
            LEFT JOIN {G}.dim_artist         da  ON e.artist_name = da.artist_name
            LEFT JOIN {G}.dim_album          dal ON TRIM(e.album_name) = dal.album_name
            LEFT JOIN {G}.dim_genre          dg  ON LOWER(TRIM(e.track_genre)) = dg.genre_name
            LEFT JOIN {G}.dim_audio_features daf ON e.track_id = daf.track_id
        )

        -- Final: generate surrogate PK and select all fact columns
        SELECT
            ROW_NUMBER() OVER (ORDER BY track_id, artist_id) AS fact_id,
            track_id,
            track_name,
            artist_id,
            album_id,
            genre_id,
            audio_id,
            popularity,
            duration_ms,
            duration_formatted,
            explicit
        FROM resolved

    ) AS source
    ON  target.track_id  = source.track_id
    AND target.artist_id = source.artist_id
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("fact_tracks MERGE completed.")

In [0]:
spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_fact_rows,
        SUM(CASE WHEN artist_id         IS NULL THEN 1 ELSE 0 END) AS missing_artist_id,
        SUM(CASE WHEN album_id          IS NULL THEN 1 ELSE 0 END) AS missing_album_id,
        SUM(CASE WHEN genre_id          IS NULL THEN 1 ELSE 0 END) AS missing_genre_id,
        SUM(CASE WHEN audio_id          IS NULL THEN 1 ELSE 0 END) AS missing_audio_id,
        SUM(CASE WHEN duration_formatted IS NULL THEN 1 ELSE 0 END) AS missing_duration_fmt
    FROM {G}.fact_tracks
""").show()

In [0]:
spark.sql(f"""
    SELECT
        track_name,
        duration_ms,
        duration_formatted
    FROM {G}.fact_tracks
    ORDER BY duration_ms ASC
    LIMIT 1000
""").show(truncate=False)

In [0]:
spark.sql(f"""
    SELECT
        da.artist_name,
        COUNT(DISTINCT ft.track_id)         AS total_tracks,
        ROUND(AVG(ft.popularity), 2)        AS avg_popularity,
        ROUND(AVG(daf.energy), 3)           AS avg_energy,
        ROUND(AVG(daf.danceability), 3)     AS avg_danceability
    FROM {G}.fact_tracks        ft
    JOIN {G}.dim_artist         da  ON ft.artist_id = da.artist_id
    JOIN {G}.dim_audio_features daf ON ft.audio_id  = daf.audio_id
    GROUP BY da.artist_name
    ORDER BY avg_popularity DESC
    LIMIT 10
""").show(truncate=False)

**Notas importantes**
- Para el dataset que elegimos, el modelado snowflake no trae mucho valor; mas bien un poco de overhead, ya que las dimensiones no se aprovechan por falta de informacion (se intento hacer API enrichment). Para este dataset lo que hubiera oensado mas optimo es una unica tabla (OBT) que separe los artistas en diferentes registros
- Se agregaron surrogate keys para poder cumplir con el snowflake shema, pero en un hipotetico caso que entrara mas .csv, el ETL se romperia ya que los IDs cambiarian